# Principal-Angle Experiment Extension

**Experiment A:** mutually disjoint rollout samples (N=1000, 500, 200), excluding the original 1000. For each rollout sample, we sample without replacement, construct our correct and incorrect representations, and compute the principal angles corresponding to the rollout sample.

**Experiment B:** mutually prompt-disjoint samples (N=1000, 500, 200), using only prompts absent from the original 1000. This is a similar experiment to A but sampling prompts without replacement instead of directly random sampling rollouts from our data.



In [8]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.utils.extmath import randomized_svd
from google.colab import drive

drive.mount("/content/gdrive")

ROOT = Path("/content/gdrive/MyDrive/gemma3_1b_it_gsm8k_raw26_v3_lean")
META = ROOT / "analysis_cache_spectral/rollout_metadata.parquet"
OLD_SAMPLE = ROOT / "principal_angles_full_reasoning_rank5_control_1000iters" / "sampled_1000_rollouts.csv"
OUT = ROOT / "principal_angle_stability_prompt_generalization"
OUT.mkdir(parents=True, exist_ok=True)

SIZES = [1000, 500, 200]
RANK, SEED, EPS = 5, 42, 1e-12


Drive already mounted at /content/gdrive; to attempt to forcibly remount, call drive.mount("/content/gdrive", force_remount=True).


## Load eligible population + original 1000

In [17]:
df = pd.read_parquet(META)
old = pd.read_csv(OLD_SAMPLE)

# eligible rollouts
df = df[
    df["correct"].notna() &
    df["boxed_token_start"].notna()
].copy()

RAW = ROOT / "full/raw_layer_outputs"

df["hidden_path"] = df.apply(
    lambda r: str(
        RAW / f"{r.sample_id}_r{int(r.rollout_id):02d}.npz"
    ),
    axis=1
)

df = df[df["hidden_path"].map(lambda p: Path(p).exists())].copy()

df["sample_id"] = df["sample_id"].astype(str)
df["rollout_id"] = df["rollout_id"].astype(str)

old["sample_id"] = old["sample_id"].astype(str)
old["rollout_id"] = old["rollout_id"].astype(str)

df["key"] = list(zip(df["sample_id"], df["rollout_id"]))
old["key"] = list(zip(old["sample_id"], old["rollout_id"]))

old_keys = set(old["key"])
old_prompts = set(old["sample_id"])

print("Eligible rollouts:", len(df))
print("Original rollouts:", len(old_keys))
print("Original prompts:", len(old_prompts))

Eligible rollouts: 9780
Original rollouts: 1000
Original prompts: 637


## Experiment A: Testing for Rollout Level Generalization.

In [18]:
print("df after filtering:", len(df))
print("original 1000:", len(old_keys))

pool_A = df[~df["key"].isin(old_keys)].copy()

print("available unseen rollouts:", len(pool_A))
print("needed:", sum(SIZES))

df after filtering: 9780
original 1000: 1000
available unseen rollouts: 8780
needed: 1700


In [19]:
pool_A = df[~df["key"].isin(old_keys)].sample(frac=1, random_state=SEED).reset_index(drop=True)
assert len(pool_A) >= sum(SIZES)

A, start = {}, 0
for n in SIZES:
    A[n] = pool_A.iloc[start:start+n].copy()
    A[n].to_csv(OUT / f"expA_rollout_disjoint_{n}.csv", index=False)
    start += n

sets_A = [set(A[n]["key"]) for n in SIZES]
assert all(not s & old_keys for s in sets_A)
for i in range(len(sets_A)):
    for j in range(i + 1, len(sets_A)):
        assert not sets_A[i] & sets_A[j]

for n in SIZES:
    y = A[n]["correct"].astype(bool)
    print(f"N={n}: correct={y.sum()}, incorrect={(~y).sum()}")


N=1000: correct=603, incorrect=397
N=500: correct=321, incorrect=179
N=200: correct=140, incorrect=60


## Experiment B: Testing for Prompt Level Generalization.

In [20]:
pool_B = df[~df["sample_id"].isin(old_prompts)].copy()

rng = np.random.default_rng(SEED)
available = set(pool_B["sample_id"].unique())
B = {}

for n in SIZES:
    order = rng.permutation(list(available))
    rows, used = [], set()

    for p in order:
        idx = pool_B.index[pool_B["sample_id"] == p].tolist()
        rng.shuffle(idx)
        rows.extend(idx)
        used.add(p)
        if len(rows) >= n:
            break

    assert len(rows) >= n
    B[n] = pool_B.loc[rows[:n]].copy()
    B[n].to_csv(OUT / f"expB_prompt_disjoint_{n}.csv", index=False)

    available -= used

sets_B = [set(B[n]["sample_id"]) for n in SIZES]
assert all(not s & old_prompts for s in sets_B)
for i in range(len(sets_B)):
    for j in range(i + 1, len(sets_B)):
        assert not sets_B[i] & sets_B[j]

for n in SIZES:
    y = B[n]["correct"].astype(bool)
    print(f"N={n}: prompts={B[n]['sample_id'].nunique()}, correct={y.sum()}, incorrect={(~y).sum()}")


N=1000: prompts=104, correct=714, incorrect=286
N=500: prompts=52, correct=279, incorrect=221
N=200: prompts=21, correct=126, incorrect=74


## Principal Angle Helpers.

In [21]:
def load_repr(row):
    with np.load(row.hidden_path) as z:
        H = z["causal_hidden_states"][:, :int(row.boxed_token_start), :].astype(np.float32)

    H /= np.maximum(np.linalg.norm(H, axis=-1, keepdims=True), EPS)
    return H.mean(axis=1)


def principal_angles(sample):
    X = np.stack([load_repr(r) for _, r in sample.iterrows()])
    y = sample["correct"].astype(bool).to_numpy()
    assert y.sum() >= RANK and (~y).sum() >= RANK

    out = []

    for layer in range(X.shape[1]):
        C = X[y, layer].copy()
        I = X[~y, layer].copy()

        C -= C.mean(0, keepdims=True)
        I -= I.mean(0, keepdims=True)

        _, _, Vc = randomized_svd(C, RANK, random_state=SEED)
        _, _, Vi = randomized_svd(I, RANK, random_state=SEED)

        s = np.linalg.svd(Vc @ Vi.T, compute_uv=False)
        theta = np.degrees(np.arccos(np.clip(s, 0, 1)))

        out += [(layer, k, a) for k, a in enumerate(theta, 1)]

    return pd.DataFrame(out, columns=["layer", "angle", "degrees"])


## Smoke test

In [22]:
def smoke_sample(df, per_class=10):
    c = df[df["correct"].astype(bool)].sample(per_class, random_state=SEED)
    i = df[~df["correct"].astype(bool)].sample(per_class, random_state=SEED)
    return pd.concat([c, i]).sample(frac=1, random_state=SEED)

for name, sample in [
    ("Experiment A", smoke_sample(A[200])),
    ("Experiment B", smoke_sample(B[200])),
]:
    test = principal_angles(sample)
    assert len(test) == 26 * RANK
    assert np.isfinite(test["degrees"]).all()
    print(f"{name}: PASS")

print("Smoke test passed.")


Experiment A: PASS
Experiment B: PASS
Smoke test passed.


## Original 1000 baseline + both experiments

In [ ]:
old_sample = old[["sample_id", "rollout_id"]].merge(
    df.drop(columns=["key"]),
    on=["sample_id", "rollout_id"],
    how="inner"
)
assert len(old_sample) == len(old)

baseline = principal_angles(old_sample)
baseline["experiment"] = "original"
baseline["n"] = len(old_sample)

results = [baseline]

for name, groups in [("rollout_disjoint", A), ("prompt_disjoint", B)]:
    for n in SIZES:
        print(f"Running {name}, N={n}")
        r = principal_angles(groups[n])
        r["experiment"] = name
        r["n"] = n
        results.append(r)

results = pd.concat(results, ignore_index=True)
results.to_csv(OUT / "principal_angles_all.csv", index=False)

print("Saved:", OUT)


## Plots

In [ ]:
for exp, title in [
    ("rollout_disjoint", "Experiment A: Rollout-Level Generalization"),
    ("prompt_disjoint", "Experiment B: Prompt-Level Generalization"),
]:
    fig, axes = plt.subplots(1, RANK, figsize=(18, 3.6), sharey=True)

    for k, ax in enumerate(axes, 1):
        b = baseline[baseline["angle"] == k]
        ax.plot(b["layer"], b["degrees"], "--", lw=2, label="Original N=1000")

        d = results[(results["experiment"] == exp) & (results["angle"] == k)]
        for n in SIZES:
            q = d[d["n"] == n]
            ax.plot(q["layer"], q["degrees"], label=f"N={n}")

        ax.set_title(f"θ{k}")
        ax.set_xlabel("Layer")
        ax.grid(alpha=0.2)

    axes[0].set_ylabel("Principal angle (°)")
    axes[-1].legend()
    fig.suptitle(title)
    plt.tight_layout()
    plt.savefig(OUT / f"{exp}.png", dpi=200, bbox_inches="tight")
    plt.show()


## Verification of Disjointness between sets/subsets.

In [ ]:
print("Experiment A")
for n in SIZES:
    print(n, "overlap with original rollouts =", len(set(A[n]["key"]) & old_keys))

print("\nExperiment B")
for n in SIZES:
    print(n, "overlap with original prompts =", len(set(B[n]["sample_id"]) & old_prompts))

print("\nAll strict disjointness assertions passed.")
